<a href="https://colab.research.google.com/github/Lekanggy/Deeplearning-class/blob/master/Transformer(Attention_is_all_you_need_paper).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

# Implementation of Self Attension(Mult-head self attension)

In [2]:
class SelfAttension(nn.Module):
 def __init__(self, embed_size, heads):
    super(SelfAttension, self).__init__()
    self.embed_size = embed_size
    self.heads = heads
    self.head_dim = embed_size // heads

    assert(self.head_dim * heads == embed_size), "Embed size needs to be div by heads"

    #Linear Transformation based on traininable parameters (Every layer generate it on weights)
    self.values = nn.Linear(embed_size, embed_size, bias=False)
    self.keys = nn.Linear(embed_size, embed_size, bias=False)
    self.queries = nn.Linear(embed_size, embed_size, bias=False)
    self.fc_out = nn.Linear(embed_size, embed_size, bias=False) # after concatenation(head_out) .

 def forward(self, values, keys, query, mask=None):
    #Look of query (N, query_len, embed_size)
    N = query.shape[0] # Number of training example

    val_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

    values = self.values(values) #(N, len, embed_size)
    keys = self.keys(keys)
    queries = self.queries(query)

    #Split the embeddings into self.head pieces(embed_size = heads * head_dim)
    values = values.reshape(N, val_len, self.heads, self.head_dim)
    keys = keys.reshape(N, key_len, self.heads, self.head_dim)
    queries = queries.reshape(N, query_len, self.heads, self.head_dim)

    # n=number of sentence, q=query_len, h=number of head, d=head_dim
    attention_score = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])

    #Mask padded indices so that their weights becomes zero
    if mask is not None:
      attention_score = attention_score.masked_fill(mask == 0, float("-1e20"))

    attention_score = torch.softmax(attention_score / (self.embed_size ** (1/2)), dim=3)

    #attension shape: (N, heads, query_len, key_len)
    # Values shape : (N, value_len, heads, head_dim)
    #out multiply: (N, query_len, heads, head_dim )

    # We reshape and concatenate the the last two dimensions
    out = torch.einsum("nhql,nlhd->nqhd", [attention_score, values]).reshape(N, query_len, self.heads * self.head_dim)

    print(f"Out attension score shape: {out.shape}")

    out = self.fc_out(out) # (N, query_len, embed_size)

    return out

In [3]:
class TransformerBlocks(nn.Module):
  def __init__(self, embed_size, heads, dropout, forward_expansion):
    super(TransformerBlocks, self).__init__()
    self.attention = SelfAttension(embed_size, heads)
    self.norm1 = nn.LayerNorm(embed_size)
    self.norm2 = nn.LayerNorm(embed_size)

    self.feed_forward = nn.Sequential(
        nn.Linear(embed_size, forward_expansion * embed_size),
        nn.ReLU(),
        nn.Linear(forward_expansion * embed_size, embed_size)
    )

    self.dropout = nn.Dropout(dropout)

  def forward(self, value, key , query, mask):
    attention = self.attention(value, key, query, mask)
    x = self.dropout(self.norm1(attention + query))
    forward = self.feed_forward(x)
    out = self.dropout(self.norm2(forward + x))
    return out

In [4]:
class Encoder(nn.Module):
  def __init__(self, src_vocab_size, embed_size, num_layers, heads, device, forward_expansion, dropout, max_length):
    super(Encoder, self).__init__()
    self.embed_size = embed_size
    self.device = device
    self.word_embedding = nn.Embedding(src_vocab_size, embed_size)
    self.position_embedding = nn.Embedding(max_length, embed_size)

    self.layers = nn.ModuleList(
        [
            TransformerBlocks(
                embed_size,
                heads,
                dropout,
                forward_expansion=forward_expansion
            )
            for _ in range(num_layers)
        ]
      )

    self.dropout = nn.Dropout(dropout)

  def forward(self, x, mask):
    N, seq_length = x.shape
    positions = torch.arange(0, seq_length).expand(N, seq_length).to(self.device)

    out = self.dropout(self.word_embedding(x) + self.position_embedding(positions))

    # The query, key and value are all the same in the encoder. It chnages in the decoder block
    for layer in self.layers:
      out = layer(out, out, out, mask)

    return out


In [5]:
class DecoderBlock(nn.Module):
  def __init__(self, embed_size, heads, forward_expansion, dropout, device):
    super(DecoderBlock, self).__init__()

    self.norm = nn.LayerNorm(embed_size)
    self.attention = SelfAttension(embed_size, heads=heads)

    self.transformer_block = TransformerBlocks(
        embed_size, heads, dropout, forward_expansion
    )

    self.dropout = nn.Dropout(dropout)

  def forward(self, x, value, key, src_mask, trg_mask):
    attention = self.attention(x, x, x, trg_mask)
    query = self.dropout(self.norm(attention + x))
    out = self.transformer_block(value, key, query, src_mask)
    return out


In [6]:
class Decoder(nn.Module):
  def __init__(self, trg_vocab_size, embed_size, num_layers, heads, forward_expansion, dropout, max_length):
    super(Decoder, self).__init__()
    self.device = device
    self.word_embedding = nn.Embedding(trg_vocab_size, embed_size)
    self.position_embedding = nn.Embedding(max_length, embed_size)

    self.layers = nn.ModuleList([DecoderBlock(embed_size, heads, forward_expansion, dropout, device) for _ in range(num_layers)])
    self.fc_out = nn.Linear(embed_size, trg_vocab_size) # Corrected output layer
    self.dropout = nn.Dropout(dropout)

  def forward(self, x, enc_out, src_mask, trg_mask):
    N, seq_length = x.shape
    positions = torch.arange(0, seq_length).expand(N, seq_length).to(self.device)

    x = self.dropout(self.word_embedding(x) + self.position_embedding(positions))

    for layer in self.layers:
      x = layer(x, enc_out, enc_out, src_mask, trg_mask) # Corrected arguments for DecoderBlock.forward

    out = self.fc_out(x)
    return out

In [7]:
class Transformer(nn.Module):
  def __init__(self, src_vocab_size, trg_vocab_size, src_pad_idx, trg_pad_idx, embed_size=512, num_layers=6, forward_expansion=4, heads=8, dropout=0, device="cpu", max_length=100):
    super(Transformer, self).__init__()
    self.encoder = Encoder(
        src_vocab_size,
        embed_size,
        num_layers,
        heads,
        device,
        forward_expansion,
        dropout,
        max_length
    )


    self.decoder = Decoder(
        trg_vocab_size,
        embed_size,
        num_layers,
        heads,
        forward_expansion,
        dropout,
        max_length
    )

    self.src_pad_idx = src_pad_idx # pad tokens
    self.trg_pad_idx = trg_pad_idx # Futuere token mask
    self.device = device

  def make_src_mask(self, src):
    src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)
    #(N, 1, 1, src_len)
    return src_mask.to(self.device)

  def make_trg_mask(self, trg):
    N, trg_len = trg.shape
    trg_mask = torch.tril(torch.ones((trg_len, trg_len))).expand(N, 1, trg_len, trg_len)
    return trg_mask.to(self.device)

  def forward(self, src, trg):
    src_mask = self.make_src_mask(src)
    trg_mask = self.make_trg_mask(trg)
    enc_src = self.encoder(src, src_mask)
    out = self.decoder(trg, enc_src, src_mask, trg_mask)
    return out


# Text the Transformer

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

x = torch.tensor([[1, 3, 4, 5, 6, 7 , 7, 8, 0], [1, 8, 9, 5, 6, 7, 2, 3, 6]],  device=device)
trg = torch.tensor([[1, 7, 8, 4, 8, 9 , 0, 1, 9], [7, 2, 3, 4, 6, 5, 6, 7, 0]], device=device)

src_pad_idx = 0
trg_pad_idx = 0
src_vocab_size = 10
trg_vocab_size = 10
model = Transformer(src_vocab_size, trg_vocab_size, src_pad_idx, trg_pad_idx, device=device).to(device)
out = model(x, trg[:, :-1])
print(out.shape)

Out attension score shape: torch.Size([2, 9, 512])
Out attension score shape: torch.Size([2, 9, 512])
Out attension score shape: torch.Size([2, 9, 512])
Out attension score shape: torch.Size([2, 9, 512])
Out attension score shape: torch.Size([2, 9, 512])
Out attension score shape: torch.Size([2, 9, 512])
Out attension score shape: torch.Size([2, 8, 512])
Out attension score shape: torch.Size([2, 8, 512])
Out attension score shape: torch.Size([2, 8, 512])
Out attension score shape: torch.Size([2, 8, 512])
Out attension score shape: torch.Size([2, 8, 512])
Out attension score shape: torch.Size([2, 8, 512])
Out attension score shape: torch.Size([2, 8, 512])
Out attension score shape: torch.Size([2, 8, 512])
Out attension score shape: torch.Size([2, 8, 512])
Out attension score shape: torch.Size([2, 8, 512])
Out attension score shape: torch.Size([2, 8, 512])
Out attension score shape: torch.Size([2, 8, 512])
torch.Size([2, 8, 10])
